### Getting the deputies data

We will work with the LVI (56th) period of the "Cámara de Diputadas y Diputados" (Deputies' Chamber), the dates of this period are from 2022-03-11 to 2026-03-10.

We want to get the deputy ID (dipid), names, gender, date of birth and political militancies.

Since 2026-03-11 starts a new cohort of deputies I will use a specific call to get the information of the deputies, instead of actual period, this will not work while working the code and also not past the 11-03

There's a difference between Periodo legislativo (4 year term of the Chamber), and the Legislatura (1year subdivision in the periodo Legislativo)

In [32]:
import requests
import sqlite3
import pandas as pd
import xml.etree.ElementTree as et
from datetime import datetime


From the database that the Biblioteca Nacional del Congreso (ADD THE LINK) we see that the Periodo legislativo that we want to work with is the officially named LVI (56th), but in the inner database the whole period is numbered with ID=10

[This is where we can find the endpoints from the Cámara del Congreso](https://www.camara.cl/transparencia/datosAbiertos.aspx)

In [33]:
# this is the endpoint URL that we will work on
url = "https://opendata.camara.cl/camaradiputados/WServices/WSDiputado.asmx/retornarDiputadosXPeriodo"

# calling the API
# the id we are usign now is 10, that's the Period 2022-2026
response = requests.get(url, params={"prmPeriodoID":10}) #add timeout?

# this is the answer we get
deputies_xml = response.text

# save the data so we don't have to download it again
with open(f"diputados_periodo_10.xml", "w", encoding="utf-8") as f:
    f.write(deputies_xml)

In [ ]:
deputies_xml

'<?xml version="1.0" encoding="utf-8"?>\r\n<DiputadosPeriodoColeccion xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns="http://opendata.camara.cl/camaradiputados/v1">\r\n  <DiputadoPeriodo>\r\n    <FechaInicio>2026-03-10T23:59:59</FechaInicio>\r\n    <FechaTermino xsi:nil="true" />\r\n    <Diputado>\r\n      <Id>1096</Id>\r\n      <Nombre>María Candelaria</Nombre>\r\n      <Nombre2 />\r\n      <ApellidoPaterno>Acevedo</ApellidoPaterno>\r\n      <ApellidoMaterno>Sáez</ApellidoMaterno>\r\n      <FechaNacimiento>1958-09-12T00:00:00</FechaNacimiento>\r\n      <RUT />\r\n      <RUTDV />\r\n      <Sexo Valor="0">Femenino</Sexo>\r\n      <Militancias>\r\n        <Militancia>\r\n          <FechaInicio>2022-03-11T00:00:00</FechaInicio>\r\n          <FechaTermino>2026-03-10T23:59:59</FechaTermino>\r\n          <Partido>\r\n            <Id>PC</Id>\r\n            <Nombre>Partido Comunista</Nombre>\r\n            <Alias>PC</Alias>\r\n         

In [ ]:
# parse XML using ElementTree, setting the data in a tree structure
deputies_parsed = et.fromstring(deputies_xml)
deputies_parsed

# since in the tree structure the namespace gets inhered fir every children and grandchildren of the root, we have to clean and reduce it
# if we want to work with the tags we need to find a way to work also with the namespace
ns = {"ns": "http://opendata.camara.cl/camaradiputados/v1"}





In [ ]:
# we create list to store the inform
deputies_df = []
parties_df = []
militancies_df = []

def clean(s):
    return s.strip() if isinstance(s, str) else s

#stripping
# pattern 1: protect against None
# text = (some_text or "").strip() or None

# # pattern 2: only strip if it exists
# text = some_text.strip() if some_text is not None else None

# loop over each deputy
for dp in deputies_parsed.findall("ns:DiputadoPeriodo", ns):
    # first the information on the term boundaries
    term_start = dp.findtext("ns:FechaInicio", default=None, namespaces=ns)
    term_end = dp.findtext("ns:FechaTermino", default=None, namespaces=ns)
    # now we know that the term_start is really the end of the electoral process

    # now nested inside there's the concrete information on the deputy
    diputado = dp.find("ns:Diputado", ns)
    if diputado is None:
        continue

    diputado_id = diputado.findtext("ns:Id", default=None, namespaces=ns)
    if diputado_id is None:
        continue
    dipid = int(diputado_id)

    # the sex element has text and attribute
    sex_element = dp.find("ns:Sexo", ns)
    sex_text = sex_element.text if sex_element is not None else None
    
    deputies_df.append({
        "dipid": dipid,
        "nombre": clean(dp.findtext("ns:Nombre", default=None, namespaces=ns)), # some deputies have two names, but they are all in the same tag
        "apellido_paterno": clean(dp.findtext("ns:ApellidoPaterno", default=None, namespaces=ns)),
        "apellido_materno": clean(dp.findtext("ns:ApellidoMaterno", default=None, namespaces=ns)),
        "fecha_nacimiento": clean(dp.findtext("ns:FechaNacimiento", default=None, namespaces=ns)),
        "sex_text": sex_text,
    })

    # now we proceeed with the information on the militancies
    militancia = dp.find("ns:Militancia", ns)
    if militancia is None:
        continue
    for i in militancia.findall("ns:Militancias", ns):
        i_start = i.findtext("ns:FechaInicio", default=None, namespaces=ns)
        i_end = i.findtext("ns:FechaTermino", default=None, namespaces=ns)

        partido = i.find("ns:Partido", ns)

        party_id = partido.findtext("ns:Id", default=None, namespaces=ns)
        party_name = partido.findtext("ns:Nombre", default=None, namespaces=ns)
        party_alias = partido.findtext("ns:Alias", default=None, namespaces=ns)

In [46]:
show =deputies_parsed.findall("ns:DiputadoPeriodo", ns)[1].findtext("ns:Id", default=None, namespaces=ns)
show

In [ ]:
deputies_parsed.get(".//ns:diputado", ns)

Deputies: 157
Parties: 29
Militancy intervals: 332

Top 10 deputies by number of militancy intervals:
dipid
74      6
917     5
872     5
1003    5
1068    5
1086    5
1082    5
1074    5
1087    5
948     4
dtype: int64


,dipid,nombre,nombre2,apellido_paterno,apellido_materno,fecha_nacimiento,sexo_valor,sexo_text
0,1096,María Candelaria,,Acevedo,Sáez,1958-09-12T00:00:00,0,Femenino
1,1097,Eric,,Aedo,Jeldres,1968-07-13T00:00:00,1,Masculino
2,1098,Yovana,,Ahumada,Palma,1973-02-24T00:00:00,0,Femenino


,party_id,party_name,party_alias
0,PC,Partido Comunista,PC
1,DC,Partido Demócrata Cristiano,DC
2,PDG,Partido de la Gente,PDG
3,IND,Independientes,IND
4,PSC,Partido Social Cristiano,PSC
5,UDI,Unión Demócrata Independiente,UDI
10,PPD,Partido Por la Democracia,PPD
12,PREP,Partido Republicano,PREP
16,PS,Partido Socialista,PS
21,RN,Renovación Nacional,RN


,dipid,party_id,date_start,date_end
0,1096,PC,2022-03-11T00:00:00,2026-03-10T23:59:59
1,1097,DC,2022-03-11T00:00:00,2026-03-10T23:59:59
2,1098,PDG,2022-03-11T00:00:00,2023-04-18T23:59:59
3,1098,IND,2023-04-19T00:00:00,2025-03-16T23:59:59
4,1098,PSC,2025-03-17T00:00:00,2026-03-10T23:59:59
5,1009,UDI,2018-03-11T00:00:00,2022-03-10T23:59:59
6,1009,UDI,2022-03-11T00:00:00,2026-03-10T23:59:59
7,803,IND,2010-03-11T00:00:00,2014-03-10T23:59:59
8,803,IND,2018-03-11T00:00:00,2022-03-10T23:59:59
9,803,IND,2022-03-11T00:00:00,2026-03-10T23:59:59



Saved to camara.db (tables: deputies, parties, deputies_militancies).


In [ ]:
deputies_parsed.findall("ns:DiputadoPeriodo", ns)[0].

'2026-03-10T23:59:59'

Here we see that the periodo we need is called "2022-2026" and it's registerd as id = 10

In [12]:
periodo_id = 10

periodo_nombre = "2022-2026"

periodo_inicio = "2022-03-11"

periodo_termino = "2026-03-10"

In [4]:
url_periodo = "https://opendata.camara.cl/camaradiputados/WServices/WSDiputado.asmx/retornarDiputadosXPeriodo?prmPeriodoID=10"


response_periodo = requests.get(url_periodo)

print(response_periodo)
print(response_periodo.status_code)

<Response [200]>
200


In [12]:
#dom = parse(response_periodo.conten)
dom = parseString(response_periodo.text)

In [18]:
dom.toprettyxml(indent=" ")

'<?xml version="1.0" ?>\n<DiputadosPeriodoColeccion xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns="http://opendata.camara.cl/camaradiputados/v1">\n \n  \n <DiputadoPeriodo>\n  \n    \n  <FechaInicio>2026-03-10T23:59:59</FechaInicio>\n  \n    \n  <FechaTermino xsi:nil="true"/>\n  \n    \n  <Diputado>\n   \n      \n   <Id>1096</Id>\n   \n      \n   <Nombre>María Candelaria</Nombre>\n   \n      \n   <Nombre2/>\n   \n      \n   <ApellidoPaterno>Acevedo</ApellidoPaterno>\n   \n      \n   <ApellidoMaterno>Sáez</ApellidoMaterno>\n   \n      \n   <FechaNacimiento>1958-09-12T00:00:00</FechaNacimiento>\n   \n      \n   <RUT/>\n   \n      \n   <RUTDV/>\n   \n      \n   <Sexo Valor="0">Femenino</Sexo>\n   \n      \n   <Militancias>\n    \n        \n    <Militancia>\n     \n          \n     <FechaInicio>2022-03-11T00:00:00</FechaInicio>\n     \n          \n     <FechaTermino>2026-03-10T23:59:59</FechaTermino>\n     \n          \n     <Parti

In [6]:
soup_periodo = BeautifulSoup(response_periodo.content, features="xml")
soup_periodo

<?xml version="1.0" encoding="utf-8"?>
<DiputadosPeriodoColeccion xmlns="http://opendata.camara.cl/camaradiputados/v1" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
<DiputadoPeriodo>
<FechaInicio>2026-03-10T23:59:59</FechaInicio>
<FechaTermino xsi:nil="true"/>
<Diputado>
<Id>1096</Id>
<Nombre>María Candelaria</Nombre>
<Nombre2/>
<ApellidoPaterno>Acevedo</ApellidoPaterno>
<ApellidoMaterno>Sáez</ApellidoMaterno>
<FechaNacimiento>1958-09-12T00:00:00</FechaNacimiento>
<RUT/>
<RUTDV/>
<Sexo Valor="0">Femenino</Sexo>
<Militancias>
<Militancia>
<FechaInicio>2022-03-11T00:00:00</FechaInicio>
<FechaTermino>2026-03-10T23:59:59</FechaTermino>
<Partido>
<Id>PC</Id>
<Nombre>Partido Comunista</Nombre>
<Alias>PC</Alias>
</Partido>
</Militancia>
</Militancias>
</Diputado>
</DiputadoPeriodo>
<DiputadoPeriodo>
<FechaInicio>2026-03-10T23:59:59</FechaInicio>
<FechaTermino xsi:nil="true"/>
<Diputado>
<Id>1097</Id>
<Nombre>Eric</Nombre>
<Nombre2/>
<Apell

Get the information into pandas df

In [ ]:
# --- Imports (all standard except pandas) ---

# --- Endpoints (official) ---
BASE = "https://opendata.camara.cl/camaradiputados/WServices"

URL_DIPUTADOS_X_PERIODO = f"{BASE}/WSDiputado.asmx/retornarDiputadosXPeriodo"
URL_DISTRITOS = f"{BASE}/WSComun.asmx/retornarDistritos"
URL_REGIONES = f"{BASE}/WSComun.asmx/retornarRegiones"
URL_PROVINCIAS = f"{BASE}/WSComun.asmx/retornarProvincias"

# --- Helpers ---
NS = {"ns": "http://opendata.camara.cl/camaradiputados/v1"}

def fetch_xml(url: str, params: dict | None = None) -> ET.Element:
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    return ET.fromstring(r.text)

def text_or_none(elem: ET.Element | None) -> str | None:
    if elem is None:
        return None
    t = (elem.text or "").strip()
    return t if t != "" else None

# --- (A) Pull deputies in period and parse core tables ---
def parse_diputados_periodo(periodo_id: int):
    root = fetch_xml(URL_DIPUTADOS_X_PERIODO, params={"prmPeriodoID": str(periodo_id)})

    deputies_rows = []
    terms_rows = []
    milit_rows = []
    parties_rows = []

    # One DiputadoPeriodo per deputy-term
    for dp in root.findall("ns:DiputadoPeriodo", NS):
        term_start = text_or_none(dp.find("ns:FechaInicio", NS))
        term_end = text_or_none(dp.find("ns:FechaTermino", NS))  # may be nil
        diputado = dp.find("ns:Diputado", NS)

        dipid = int(text_or_none(diputado.find("ns:Id", NS)))
        nombre = text_or_none(diputado.find("ns:Nombre", NS))
        nombre2 = text_or_none(diputado.find("ns:Nombre2", NS))
        ap_pat = text_or_none(diputado.find("ns:ApellidoPaterno", NS))
        ap_mat = text_or_none(diputado.find("ns:ApellidoMaterno", NS))
        fnac = text_or_none(diputado.find("ns:FechaNacimiento", NS))

        sexo_elem = diputado.find("ns:Sexo", NS)
        sexo_text = text_or_none(sexo_elem)
        sexo_val = sexo_elem.attrib.get("Valor") if sexo_elem is not None else None

        deputies_rows.append({
            "dipid": dipid,
            "nombre": nombre,
            "nombre2": nombre2,
            "apellido_paterno": ap_pat,
            "apellido_materno": ap_mat,
            "fecha_nacimiento": fnac,
            "sexo_valor": sexo_val,
            "sexo_text": sexo_text
        })

        # Term row (district often appears later in the full XML; add when you locate the tag)
        terms_rows.append({
            "dipid": dipid,
            "periodo_id": periodo_id,
            "term_start": term_start,
            "term_end": term_end,
            "district_num": None  # TODO: fill once you find the district element name/path
        })

        # Militancias (history)
        militancias = diputado.find("ns:Militancias", NS)
        if militancias is not None:
            for m in militancias.findall("ns:Militancia", NS):
                ms = text_or_none(m.find("ns:FechaInicio", NS))
                me = text_or_none(m.find("ns:FechaTermino", NS))
                partido = m.find("ns:Partido", NS)
                if partido is None:
                    continue

                party_id = text_or_none(partido.find("ns:Id", NS))
                party_name = text_or_none(partido.find("ns:Nombre", NS))
                party_alias = text_or_none(partido.find("ns:Alias", NS))

                parties_rows.append({
                    "party_id": party_id,
                    "party_name": party_name,
                    "party_alias": party_alias
                })

                milit_rows.append({
                    "dipid": dipid,
                    "party_id": party_id,
                    "date_start": ms,
                    "date_end": me
                })

    df_deps = pd.DataFrame(deputies_rows).drop_duplicates(subset=["dipid"])
    df_terms = pd.DataFrame(terms_rows).drop_duplicates(subset=["dipid", "periodo_id"])
    df_parties = pd.DataFrame(parties_rows).drop_duplicates(subset=["party_id"])
    df_milit = pd.DataFrame(milit_rows).drop_duplicates(subset=["dipid", "party_id", "date_start"])

    return df_deps, df_terms, df_parties, df_milit

# --- (B) Double-check: do deputies have >1 militancy interval? (switchers exist) ---
def quick_militancy_audit(df_milit: pd.DataFrame, top_n: int = 10):
    counts = (df_milit.groupby("dipid")
                    .size()
                    .sort_values(ascending=False)
                    .head(top_n))
    return counts

# --- (C) Districts and region/comuna official data ---
def parse_distritos():
    root = fetch_xml(URL_DISTRITOS)
    rows = []
    for d in root.findall("ns:Distrito", NS):
        district_num = int(text_or_none(d.find("ns:Numero", NS)))
        comunas = d.find("ns:Comunas", NS)
        if comunas is None:
            continue
        for c in comunas.findall("ns:Comuna", NS):
            comuna_num = int(text_or_none(c.find("ns:Numero", NS)))
            comuna_name = text_or_none(c.find("ns:Nombre", NS))
            rows.append({"district_num": district_num, "comuna_num": comuna_num, "comuna_name": comuna_name})
    return pd.DataFrame(rows).drop_duplicates()

def parse_regiones_provincias_comunas():
    # Prefer regiones: region -> provincias -> comunas (if comunas are included in actual response)
    root = fetch_xml(URL_REGIONES)
    rows = []
    for r in root.findall("ns:Region", NS):
        region_num = int(text_or_none(r.find("ns:Numero", NS)))
        region_name = text_or_none(r.find("ns:Nombre", NS))
        provs = r.find("ns:Provincias", NS)
        if provs is None:
            continue
        for p in provs.findall("ns:Provincia", NS):
            prov_num = int(text_or_none(p.find("ns:Numero", NS)))
            prov_name = text_or_none(p.find("ns:Nombre", NS))
            comunas = p.find("ns:Comunas", NS)
            if comunas is None:
                continue
            for c in comunas.findall("ns:Comuna", NS):
                comuna_num = int(text_or_none(c.find("ns:Numero", NS)))
                comuna_name = text_or_none(c.find("ns:Nombre", NS))
                rows.append({
                    "comuna_num": comuna_num,
                    "comuna_name": comuna_name,
                    "province_num": prov_num,
                    "province_name": prov_name,
                    "region_num": region_num,
                    "region_name": region_name
                })
    return pd.DataFrame(rows).drop_duplicates()

# --- (D) SQLite schema creation (composite PK for militancies) ---
def create_sqlite(db_path="camara.db"):
    con = sqlite3.connect(db_path)
    cur = con.cursor()

    cur.executescript("""
    PRAGMA foreign_keys = ON;

    CREATE TABLE IF NOT EXISTS deputies (
        dipid INTEGER PRIMARY KEY,
        nombre TEXT,
        nombre2 TEXT,
        apellido_paterno TEXT,
        apellido_materno TEXT,
        fecha_nacimiento TEXT,
        sexo_valor TEXT,
        sexo_text TEXT
    );

    CREATE TABLE IF NOT EXISTS parties (
        party_id TEXT PRIMARY KEY,
        party_name TEXT,
        party_alias TEXT
    );

    CREATE TABLE IF NOT EXISTS deputies_militancies (
        dipid INTEGER NOT NULL,
        party_id TEXT NOT NULL,
        date_start TEXT NOT NULL,
        date_end TEXT,
        PRIMARY KEY (dipid, party_id, date_start),
        FOREIGN KEY (dipid) REFERENCES deputies(dipid),
        FOREIGN KEY (party_id) REFERENCES parties(party_id)
    );

    CREATE TABLE IF NOT EXISTS deputy_terms (
        dipid INTEGER NOT NULL,
        periodo_id INTEGER NOT NULL,
        term_start TEXT,
        term_end TEXT,
        district_num INTEGER,
        PRIMARY KEY (dipid, periodo_id),
        FOREIGN KEY (dipid) REFERENCES deputies(dipid)
    );

    CREATE TABLE IF NOT EXISTS districts (
        district_num INTEGER PRIMARY KEY,
        region_num INTEGER,
        region_name TEXT
    );

    CREATE TABLE IF NOT EXISTS district_comunas (
        district_num INTEGER NOT NULL,
        comuna_num INTEGER NOT NULL,
        comuna_name TEXT,
        PRIMARY KEY (district_num, comuna_num),
        FOREIGN KEY (district_num) REFERENCES districts(district_num)
    );

    CREATE TABLE IF NOT EXISTS comunas_lookup (
        comuna_num INTEGER PRIMARY KEY,
        comuna_name TEXT,
        province_num INTEGER,
        province_name TEXT,
        region_num INTEGER,
        region_name TEXT
    );
    """)
    con.commit()
    return con

# Example usage (minimal):
# periodo_id = 10
# df_deps, df_terms, df_parties, df_milit = parse_diputados_periodo(periodo_id)
# print(quick_militancy_audit(df_milit))
# con = create_sqlite("camara.db")
# df_deps.to_sql("deputies", con, if_exists="append", index=False)
# df_parties.to_sql("parties", con, if_exists="append", index=False)
# df_milit.to_sql("deputies_militancies", con, if_exists="append", index=False)
# df_terms.to_sql("deputy_terms", con, if_exists="append", index=False)

In [ ]:
# get the xml
